# Training MetricGAN+KAN
Mainly based on train.py from MetricGAN+. history

In [ ]:
!pip install speechbrain
!pip install hyperpyyaml
!pip install pesq
!pip install pysepm
!pip install git+https://github.com/schmiph2/pysepm.git
!pip install tensorboard

In [ ]:
import os
import copy
import shutil
import sys
from enum import Enum, auto

import torch
import torchaudio
from hyperpyyaml import load_hyperpyyaml
from pesq import pesq

import speechbrain as sb
from speechbrain.dataio.sampler import ReproducibleWeightedRandomSampler
from speechbrain.nnet.loss.stoi_loss import stoi_loss
from speechbrain.processing.features import spectral_magnitude
from speechbrain.utils.distributed import run_on_main
from speechbrain.utils.metric_stats import MetricStats

# Ensure cluster notebooks import the local project module, not another clone.
repo_root = os.getcwd()
local_train_py = os.path.join(repo_root, "train.py")
if not os.path.isfile(local_train_py):
    raise FileNotFoundError(
        f"Expected local train.py in current working directory, got: {repo_root}"
    )
if repo_root in sys.path:
    sys.path.remove(repo_root)
sys.path.insert(0, repo_root)

# from MetricGAN_KAN import MetricDiscriminator, EnhancementGenerator

from pysepm import composite
import train as train_module
print("Using train module:", train_module.__file__)
from train import *
from spectrogram import *

N_fft = 512
data_folder = "../data/noisy-vctk-16k"
output_folder = "./results"
figures_folder = "./figures"

In [ ]:
class PerceptualDistillation:
    def __init__(self, generator_old, sample_rate=16000):
        self.generator_old = generator_old
        self.generator_old.eval()
        for p in self.generator_old.parameters():
            p.requires_grad_(False)
        self.sample_rate = sample_rate

    def distillation_loss(self, noisy_wavs, lens, new_generator, hparams):
        # Feature pipeline identical to MGKBrain.compute_feats
        feats = hparams.compute_STFT(noisy_wavs)
        feats = spectral_magnitude(feats, power=0.5)
        feats = torch.log1p(feats)

        with torch.no_grad():
            mask_old = self.generator_old(feats, lengths=lens)
            mask_old = mask_old.clamp(min=hparams.min_mask).squeeze(2)
            spec_old = torch.mul(mask_old, feats)
            wav_old = hparams.resynth(torch.expm1(spec_old), noisy_wavs).detach()

        mask_new = new_generator(feats, lengths=lens)
        mask_new = mask_new.clamp(min=hparams.min_mask).squeeze(2)
        spec_new = torch.mul(mask_new, feats)
        wav_new = hparams.resynth(torch.expm1(spec_new), noisy_wavs)

        return stoi_loss(wav_new, wav_old, lens)

In [ ]:
class MGKBrainPD(MGKBrain):
    def __init__(self, *args, pd_lambda=0.5, pd_anchor_interval=50, **kwargs):
        super().__init__(*args, **kwargs)
        self.pd_lambda = pd_lambda
        self.pd_anchor_interval = pd_anchor_interval

        # Persistenter LwF-Anchor, damit Resume aus Checkpoint den Teacher-Zustand behält.
        self.generator_anchor = copy.deepcopy(self.modules.generator)
        self.generator_anchor.eval()
        for p in self.generator_anchor.parameters():
            p.requires_grad_(False)

        if self.checkpointer is not None:
            self.checkpointer.add_recoverable("generator_anchor", self.generator_anchor)

        self.perceptual_distillation = PerceptualDistillation(self.generator_anchor)

    def _refresh_anchor_from_current(self):
        self.generator_anchor.load_state_dict(self.modules.generator.state_dict())
        self.generator_anchor.eval()
        for p in self.generator_anchor.parameters():
            p.requires_grad_(False)

    def on_stage_start(self, stage, epoch=None):
        super().on_stage_start(stage, epoch)
        if stage == sb.Stage.TRAIN and self.pd_lambda > 0:
            should_refresh = False

            # Wenn kein fixer Intervall gesetzt ist, nur initial refreshen.
            if self.pd_anchor_interval <= 0:
                should_refresh = epoch is None and self.perceptual_distillation is None

            # Reguläres Anchor-Refresh pro Intervall.
            elif epoch is not None:
                should_refresh = (
                    epoch > 1 and epoch % self.pd_anchor_interval == 0
                )

            if should_refresh:
                self._refresh_anchor_from_current()

            if self.perceptual_distillation is None:
                self.perceptual_distillation = PerceptualDistillation(
                    self.generator_anchor
                )

    def compute_objectives(self, predictions, batch, stage, optim_name=""):
        loss = super().compute_objectives(predictions, batch, stage, optim_name=optim_name)
        if (
            stage == sb.Stage.TRAIN
            and self.sub_stage == SubStage.GENERATOR
            and self.pd_lambda > 0
            and self.perceptual_distillation is not None
        ):
            noisy_wavs, lens = batch.noisy_sig
            pd_loss = self.perceptual_distillation.distillation_loss(
                noisy_wavs, lens, self.modules.generator, self.hparams
            )
            self.hparams.train_logger.log_stats({"PD loss (unscaled)": pd_loss.item()})
            loss = loss + self.pd_lambda * pd_loss
        return loss

# Requirements
```bash
pip install speechbrain
pip install https://github.com/schmiph2/pysepm/archive/master.zip
```

If `kaiser` is not found leading to an `ImportError`, you may need to go to line 2 of /path/to/your/python_env/site-packages/pysepm/utils.py, and change
```python
from scipy.signal import firls,kaiser,upfirdn
```
to
```python
from scipy.signal import firls,upfirdn
from scipy.signal.windows import kaiser
```

# Training
Mainly based on train.py from MetricGAN+.

In [ ]:
def train_and_eval(hparams_file, pd_lambda=None, pd_anchor_interval=None):
    # hparams_file, run_opts, _ = sb.parse_arguments(model_param_file)
    run_opts = {"device": "cuda:3"}
    model_name = hparams_file.split(".")[-2]
    overrides = {"output_folder": os.path.join(output_folder, model_name), "data_folder": data_folder, "number_of_epochs": 120}
    with open(hparams_file) as fin:
        hparams = load_hyperpyyaml(fin, overrides=overrides)

    if pd_lambda is None:
        pd_lambda = hparams.get("pd_lambda", 0.5)
    if pd_anchor_interval is None:
        pd_anchor_interval = hparams.get("pd_anchor_interval", 50)

    # Initialize ddp (useful only for multi-GPU DDP training)
    sb.utils.distributed.ddp_init_group(run_opts)

    # Data preparation
    from voicebank_prepare import prepare_voicebank  # noqa

    run_on_main(
        prepare_voicebank,
        kwargs={
            "data_folder": hparams["data_folder"],
            "save_folder": hparams["data_folder"],
            "skip_prep": hparams["skip_prep"],
        },
    )

    # Create dataset objects
    datasets = dataio_prep(hparams)

    # Create experiment directory
    sb.create_experiment_directory(
        experiment_directory=hparams["output_folder"],
        hyperparams_to_save=hparams_file,
        overrides=overrides,
    )

    if hparams["use_tensorboard"]:
        from speechbrain.utils.train_logger import TensorboardLogger

        hparams["tensorboard_train_logger"] = TensorboardLogger(
            hparams["tensorboard_logs"]
        )

    # Create the folder to save enhanced files (+ support for DDP)
    run_on_main(create_folder, kwargs={"folder": hparams["enhanced_folder"]})

    se_brain = MGKBrainPD(
        modules=hparams["modules"],
        hparams=hparams,
        run_opts=run_opts,
        checkpointer=hparams["checkpointer"],
        pd_lambda=pd_lambda,
        pd_anchor_interval=pd_anchor_interval,
    )
    se_brain.train_set = datasets["train"]
    se_brain.batch_size = hparams["dataloader_options"]["batch_size"]
    se_brain.sub_stage = SubStage.GENERATOR
    se_brain.historical_set = {}
    se_brain.noisy_scores = {}
    if not os.path.isfile(hparams["historical_file"]):
        if os.path.isdir(hparams["MetricGAN_KAN_folder"]):
            shutil.rmtree(hparams["MetricGAN_KAN_folder"])
    run_on_main(create_folder, kwargs={"folder": hparams["MetricGAN_KAN_folder"]})
    se_brain.load_history()

    # Load latest checkpoint to resume training
    se_brain.fit(
        epoch_counter=se_brain.hparams.epoch_counter,
        train_set=datasets["train"],
        valid_set=datasets["valid"],
        train_loader_kwargs=hparams["dataloader_options"],
        valid_loader_kwargs=hparams["valid_dataloader_options"],
    )

    # Load best checkpoint for evaluation
    test_stats = se_brain.evaluate(
        test_set=datasets["test"],
        max_key=hparams["target_metric"],
        test_loader_kwargs=hparams["dataloader_options"],
    )

    # Show parameters summary
    param_count = sum(p.numel() for p in se_brain.modules.generator.parameters() if p.requires_grad)
    se_brain.hparams.train_logger.log_stats({"Generator parameter count": param_count})
    param_count = sum(p.numel() for p in se_brain.modules.discriminator.parameters() if p.requires_grad)
    se_brain.hparams.train_logger.log_stats({"Discriminator parameter count": param_count})

    return se_brain, datasets

Loop through the model files and train them.

In [ ]:
task_yamls = ["hparams/train_mgk_g4_d4-l025.yaml", "hparams/train_mgk_g4_d4-l075.yaml", "hparams/train_mgk_g4_d4-l1.yaml"]

for idx, yaml_file in enumerate(task_yamls):
    print(f"Training: {yaml_file}")
    brain, datasets = train_and_eval(yaml_file)

In [ ]:
import glob
import re
import matplotlib.pyplot as plt


def _extract_metric(line, keys):
    for key in keys:
        m = re.search(rf"{key}\s*[:=]\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)", line, flags=re.IGNORECASE)
        if m:
            return float(m.group(1))
    return None


def load_training_curve(log_file):

    epochs, valid_pesq, valid_csig, valid_cbak, valid_covl = [], [], [], [], []

    with open(log_file, "r", encoding="utf-8") as f:
        for line in f:
            if "epoch" not in line.lower():
                continue

            m_epoch = re.search(r"epoch\s*[:=]\s*(\d+)", line, flags=re.IGNORECASE)
            if not m_epoch:
                continue

            epochs.append(int(m_epoch.group(1)))
            valid_pesq.append(_extract_metric(line, [r"pesq", r"valid[_\s-]*pesq"]))
            valid_csig.append(_extract_metric(line, [r"csig", r"valid[_\s-]*csig"]))
            valid_cbak.append(_extract_metric(line, [r"cbak", r"valid[_\s-]*cbak"]))
            valid_covl.append(_extract_metric(line, [r"covl", r"valid[_\s-]*covl"]))

    return {
        "epoch": epochs,
        "valid_pesq": valid_pesq,
        "valid_csig": valid_csig,
        "valid_cbak": valid_cbak,
        "valid_covl": valid_covl,
    }


candidates = sorted(glob.glob(f"{output_folder}/**/train_log.txt", recursive=True))
if not candidates:
    raise FileNotFoundError("Kein train_log.txt gefunden. Trainiere zuerst mindestens 1 Run.")

log_file = candidates[-1]
curves = load_training_curve(log_file)

if len(curves["epoch"]) == 0:
    raise RuntimeError(f"Keine Epochenzeilen in {log_file} gefunden.")

fig, axes = plt.subplots(2, 2, figsize=(12, 8), dpi=120)
plots = [
    ("valid_pesq", "Valid PESQ"),
    ("valid_csig", "Valid CSIG"),
    ("valid_cbak", "Valid CBAK"),
    ("valid_covl", "Valid COVL"),
]

for ax, (key, title) in zip(axes.flat, plots):
    y = curves[key]
    x = [e for e, v in zip(curves["epoch"], y) if v is not None]
    y = [v for v in y if v is not None]

    if len(x) > 0:
        ax.plot(x, y, marker="o", linewidth=1.5, markersize=3)
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.grid(True, alpha=0.3)

plt.suptitle(f"Training Curves ({os.path.dirname(log_file)})")
plt.tight_layout()

os.makedirs(figures_folder, exist_ok=True)
out_png = os.path.join(figures_folder, "training_curves_latest.png")
plt.savefig(out_png, bbox_inches="tight")
plt.show()

print(f"Log-Datei: {log_file}")
print(f"Diagramm gespeichert: {out_png}")

Generate spectrograms.

In [ ]:
model_name_list = [fname.split(".")[-2] for fname in os.listdir("models") if fname.endswith(".yaml")]
enh_folder = [f"{output_folder}/{fname.split(".")[-2]}/enhanced_wavs" for fname in os.listdir("models") if fname.endswith(".yaml")]
# os.path.join(output_folder, model_name)
for m in model_name_list:
    target_dir = f"{figures_folder}/{m}"
    if not os.path.exists(target_dir):
        os.makedirs(target_dir)
    save_spec(f"{output_folder}/{m}/enhanced_wavs", target_dir)

Codes for generating model's hyper-param files.

In [ ]:
def generate_train_config():
    templates = None
    with open("hparams/train.yaml") as f:
        templates = f.readlines()
    for fname in os.listdir("models"):
        tmp = fname.split(".")
        if tmp[-1] == "yaml":
            with open(f"hparams/train_{tmp[-2]}.yaml", "w") as f:
                spec = templates.copy()
                spec[64] = f"models: !include:../models/{fname}\n"
                f.write("# Auto generated basing on train.yaml\n")
                f.writelines(spec)
            
# generate_train_config()